# Настройка репозитория

In [ ]:
!sudo apt-get update || true
!sudo apt-get install -y software-properties-common || true

# включаем universe + multiverse
!sudo add-apt-repository -y universe || true
!sudo add-apt-repository -y multiverse || true

## Установка зависимостей

Установка инструментов разработки и библиотек для сборки проектов:

- Компиляторы (gcc, g++)
- Система сборки (CMake)
- Графические библиотеки (Qt6, QWT)
- Вспомогательные утилиты

In [ ]:
!sudo apt install cmake cmake-gui gcc g++ git libboost-all-dev libncurses5-dev libpython3-dev libxerces-c-dev libxml2 libxml2-utils libxslt1-dev python3-numpy qt6-base-dev python3-docopt openssl -y

## Графические библиотеки

Установка OpenGL-зависимостей для визуализации.

In [ ]:
!sudo apt install libglu1-mesa-dev freeglut3-dev mesa-common-dev -y

## Ставим GDAL

In [ ]:
!sudo apt install gdal-bin libgdal-dev -y

## Настройка путей

Добавление путей к заголовочным файлам GDAL для компиляции.

In [ ]:
!export CPLUS_INCLUDE_PATH=/usr/include/gdal
!export C_INCLUDE_PATH=/usr/include/gdal

# Установка Qt6

Установка инструментов Qt6 и компилятора для сборки проекта.

In [ ]:
%%bash
set -euo pipefail

# 0) Инструменты Qt6 (для qmake6) + компилятор/гит
sudo apt-get update -y
sudo apt-get install -y qt6-base-dev qt6-base-dev-tools build-essential git

# Компиляция QWT

Сборка библиотек QWT из исходного кода (так как готовые пакеты для Qt6 недоступны).

## переходим в директорию QT6

In [ ]:
import os 
os.chdir("/home/jovyan/work/LISEM/qwt-multiaxes-qt6")

## Настройка QWT

Подготовка репозитория QWT с многокоординатными осями для работы с Qt6.

In [ ]:
%cd "/home/jovyan/work/LISEM/qwt-multiaxes-qt6"
!git config --global --add safe.directory '/home/jovyan/work/LISEM/qwt-multiaxes-qt6'
!git checkout qwt-multiaxes


## Сборка QWT

Компиляция библиотек QWT с помощью qmake6 и make.

In [ ]:
!qmake6 qwt.pro
!make -j"$(nproc)"

## Установка QWT

Копирование скомпилированных библиотек QWT в системные директории.

In [ ]:
!sudo make install

## Настройка путей

Добавление пути к библиотекам QWT в системный конфиг для динамической линковки.

In [ ]:
!echo "/usr/local/qwt-6.4.0-ma/lib" | sudo tee /etc/ld.so.conf.d/qwt.conf

## Обновление кэша библиотек

Обновление системного кэша библиотек и проверка наличия QWT.

In [ ]:
!sudo ldconfig
!ldconfig -p | grep qwt
!ls -la /usr/local/qwt-6.4.0-ma/lib/libqwt*

## Добавить переменные среды

In [ ]:
!export LD_LIBRARY_PATH="/usr/local/qwt-6.4.0-ma/lib:$LD_LIBRARY_PATH"
!export PKG_CONFIG_PATH="/usr/local/qwt-6.4.0-ma/lib/pkgconfig:$PKG_CONFIG_PATH"

In [ ]:
!sudo ldconfig
!ldconfig -p | grep -i qwt || true

## Проверка установки

Проверка наличия библиотек и заголовочных файлов QWT после установки.

In [ ]:
!ls -l /usr/local/qwt-6.4.0-ma/lib/libqwt.so
!ls -1 /usr/local/qwt-6.4.0-ma/include | head

# Ставим LISEM

In [ ]:
os.chdir("/home/jovyan/work/LISEM")

## Клонирование LISEM

Клонирование основной ветки проекта LISEM для последующей сборки.

In [ ]:
"""
%%bash
set -euo pipefail
cd "/home/jovyan/work/LISEM"

# если папка уже есть — удалим, чтобы не мешала (опционально)
rm -rf lisem

# клонируем конкретно ветку main_C
git clone --branch main_C --depth 1 https://github.com/vjetten/openlisem.git lisem # можно закомментить, если ветка уже установлена 
"""

## Подготовка к сборке

Создание директории для бинарных файлов LISEM.

In [ ]:
%%bash
set -euo pipefail
cd "/home/jovyan/work/LISEM"
mkdir -p lisem-bin
cd lisem-bin

## Если нет Cmake

In [ ]:
%%bash
set -euo pipefail

# 0) установить cmake (и то, что часто нужно для Qt6)
sudo apt-get update -y
sudo apt-get install -y cmake build-essential ninja-build \
                        qt6-base-dev qt6-tools-dev qt6-tools-dev-tools
cmake --version

## Проверка QWT

Проверка наличия заголовочных файлов и библиотек QWT после установки.

In [ ]:
%%bash
ls -l /usr/local/qwt-6.4.0-ma/include/qwt_plot.h
ls -l /usr/local/qwt-6.4.0-ma/lib/libqwt.so

## Создание симлинков

Создание символических ссылок для совместимости с путями, которые ожидает LISEM.

In [ ]:
%%bash
set -euo pipefail

# подложим "старый" путь как ссылки на Qt6-версию
sudo mkdir -p /usr/local/qwt-6.4.0-ma
sudo ln -sfn /usr/local/qwt-6.4.0-ma-qt6/include /usr/local/qwt-6.4.0-ma/include
sudo ln -sfn /usr/local/qwt-6.4.0-ma-qt6/lib     /usr/local/qwt-6.4.0-ma/lib
sudo ldconfig

# проверим, что по старому пути всё видно
ls -l /usr/local/qwt-6.4.0-ma/include/qwt_plot.h
ls -l /usr/local/qwt-6.4.0-ma/lib/libqwt.so || true
readlink -f /usr/local/qwt-6.4.0-ma/lib/libqwt.so || true

# Сборка LISEM

Конфигурация и компиляция проекта LISEM с указанием путей к QWT и GDAL.

In [ ]:
%%bash
set -euo pipefail
cd "/home/jovyan/work/LISEM/lisem-bin"

# Re-run CMake configure. We also pass Qwt include/lib and GDAL so it compiles cleanly.
cmake -S ../lisem -B . -G "Unix Makefiles" \
  -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_PREFIX_PATH="/usr/lib/cmake/Qt6" \
  -DCMAKE_CXX_FLAGS="-I/usr/local/qwt-6.4.0-ma/include" \
  -DCMAKE_EXE_LINKER_FLAGS="-L/usr/local/qwt-6.4.0-ma/lib" \
  -DGDAL_INCLUDE_DIR="/usr/include/gdal" \
  -DGDAL_LIBRARY="/usr/lib/x86_64-linux-gnu/libgdal.so"

# Build with a safe job count (avoid OOM)
cmake --build . -j4


## Проверка зависимостей

Проверка динамических библиотек LISEM - поиск связей с QWT, Qt6 и GDAL.

In [ ]:
!ldd "/home/jovyan/work/LISEM/lisem-bin/Lisem" | egrep 'qwt|Qt6|gdal'

## Поиск проблемных плагинов

Поиск и удаление GRASS-плагинов GDAL, которые могут вызывать конфликты.

In [ ]:
%%bash
set -euo pipefail

# See which GRASS plugin files you have and which package owns them
ls -l /usr/lib/x86_64-linux-gnu/gdalplugins | grep -i grass || true
for f in /usr/lib/x86_64-linux-gnu/gdalplugins/*GRASS*.so; do
  [ -e "$f" ] && echo "$f -> $(dpkg -S "$f")"
done || true

# Remove the offending package(s) (try both names safely)
sudo apt-get remove -y gdal-grass || true
sudo apt-get remove -y libgdal-grass || true

## Тестовый запуск LISEM

Пробный запуск приложения на виртуальном сервере для проверки работоспособности (должно вернуть stopped (expected))

In [ ]:
%%bash
set -eo pipefail
APP="/home/jovyan/work/LISEM/lisem-bin/Lisem"
sudo apt-get -y install xvfb >/dev/null
XDG_RUNTIME_DIR="$(mktemp -d)"; chmod 700 "$XDG_RUNTIME_DIR"; export XDG_RUNTIME_DIR
LIBGL_ALWAYS_SOFTWARE=1 timeout 5 xvfb-run -a "$APP" || echo "stopped (expected)"


# Запуск расчета в LISEM

Запуск гидрологической модели LISEM с run-файлом.

In [ ]:
2+2

In [ ]:
%%bash
set -euo pipefail
APP="/home/jovyan/work/LISEM/lisem-bin/Lisem" #путь до бинарника LISEM
RUNF="/home/jovyan/work/LISEM/runfiles/Gorsch.run" # путь до runfile

# keep locale fixed (decimal dot), and silence GRASS plugin warnings
export LC_ALL=C.UTF-8 LANG=C.UTF-8 LC_NUMERIC=C.UTF-8 # Всякое форматирование 
export GDAL_SKIP=GRASS OGR_SKIP=GRASS # Отключаются проблемные GRASS-плагины GDAL
export LIBGL_ALWAYS_SOFTWARE=1 #Включается программная отрисовка OpenGL
#
# stream progress to the notebook
stdbuf -oL "$APP" -ni -r "$RUNF"


In [ ]:
!pip install GDAL

In [ ]:
os.getcwd()

In [ ]:
!find openlisem_bmi -maxdepth 2 -type f

In [ ]:
!mkdir -p openlisem-bmi-package


# Визуализация результатов

In [ ]:
import os
from osgeo import gdal
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

resdir = "/home/jovyan/work/LISEM/results/"
fnames = ["soilloss.map"]

def open_map(path):
    ds = gdal.Open(path, gdal.GA_ReadOnly)
    arr = ds.GetRasterBand(1).ReadAsArray()
    ndv = ds.GetRasterBand(1).GetNoDataValue()
    if ndv is not None:
        arr = np.ma.masked_equal(arr, ndv)
    return arr

# интервалы как в QGIS
bounds = [-9999, -10, 0, 2.5, 5, 7.5, 10, 15, 20, 9999]  
labels = [
    ">10",
    "0–10",
    "0–2.5",
    "2.5–5",
    "5–7.5",
    "7.5–10",
    "10–15",
    "15–20",
    ">20 аккумуляция"
]

# подбираем цвета (из QGIS скрина)
colors = [
    "#08306b",  # >10 аккумуляция (синий)
    "#7fcdbb",  # 0-10 аккумуляция (зелёный)
    "#f7fcb9",  # 0-2.5 (светло-жёлтый)
    "#fee08b",  # 2.5-5
    "#fdae61",  # 5-7.5
    "#f46d43",  # 7.5-10
    "#d73027",  # 10-15
    "#a50026",  # 15-20
    "#67001f"   # >20
]

cmap = ListedColormap(colors)
norm = BoundaryNorm(bounds, len(colors))

for fname in fnames:
    path = os.path.join(resdir, fname)
    if not os.path.exists(path):
        print(f"Missing: {path}")
        continue

    arr = open_map(path)
    if arr is None:
        continue

    plt.figure(figsize=(12, 10))
    im = plt.imshow(arr, cmap=cmap, norm=norm, origin="upper")
    cbar = plt.colorbar(im, shrink=0.8, ticks=[-5, 0, 1, 3.5, 6.25, 8.75, 12.5, 17.5, 25])
    cbar.ax.set_yticklabels(labels)
    plt.title(fname)
    plt.axis("off")
    plt.show()


## skip BMI

In [ ]:
!git clone --branch claude/bmi-Opus https://github.com/pihchikk/openlisem_bmi.git

In [ ]:
import os
from osgeo import gdal
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

import ipywidgets as widgets
from IPython.display import display, clear_output

# =========================
# 1. Load map as before
# =========================
resdir = "/home/jovyan/work/LISEM/results/"
fname = "soilloss.map"

def open_map(path):
    ds = gdal.Open(path, gdal.GA_ReadOnly)
    arr = ds.GetRasterBand(1).ReadAsArray()
    ndv = ds.GetRasterBand(1).GetNoDataValue()
    if ndv is not None:
        arr = np.ma.masked_equal(arr, ndv)
    return arr

bounds = [-9999, -10, 0, 2.5, 5, 7.5, 10, 15, 20, 9999]
labels = [
    ">10", "0–10", "0–2.5", "2.5–5", "5–7.5",
    "7.5–10", "10–15", "15–20", ">20 аккумуляция"
]

colors = [
    "#08306b", "#7fcdbb", "#f7fcb9", "#fee08b",
    "#fdae61", "#f46d43", "#d73027", "#a50026", "#67001f"
]

cmap = ListedColormap(colors)
norm = BoundaryNorm(bounds, len(colors))

path = os.path.join(resdir, fname)
if not os.path.exists(path):
    raise FileNotFoundError(f"Missing: {path}")

arr = open_map(path)
if isinstance(arr, np.ma.MaskedArray):
    mask = arr.mask
    target_map = arr.filled(0.0).astype(float)
else:
    mask = None
    target_map = arr.astype(float)

ny, nx = target_map.shape

# =========================
# 2. Assume *annual* rainfall used by LISEM
#    If unknown, set to typical ~1000 mm/year
# =========================
ANNUAL_RAIN_MM = 1000.0  # you can change to actual value

# =========================
# 3. Simulation state
# =========================
state = {
    "target": target_map,
    "current_map": np.zeros_like(target_map),
    "current_step": 0,
    "n_steps": 52,
    "cumulative_rain_mm": 0.0
}

# =========================
# 4. Widgets
# =========================
rain_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=100.0,
    step=1.0,
    description="Осадков (мм/неделю):",
    continuous_update=False,
    layout=widgets.Layout(width="400px")
)

step_button = widgets.Button(
    description="▶ Step",
    button_style="success",
    layout=widgets.Layout(width="120px")
)

step10_button = widgets.Button(
    description="▶▶ +10",
    button_style="info",
    layout=widgets.Layout(width="120px")
)

reset_button = widgets.Button(
    description="🔄 Reset",
    button_style="warning",
    layout=widgets.Layout(width="120px")
)

info_label = widgets.HTML(
    value="<b>Неделя:</b> 0 | <b>Осадков всего:</b> 0 mm"
)

output_area = widgets.Output()

# =========================
# 5. Plotting
# =========================
def update_plot():
    with output_area:
        clear_output(wait=True)

        img = state["current_map"]
        if mask is not None:
            img = np.ma.array(img, mask=mask)

        fig, ax = plt.subplots(figsize=(8, 8))
        im = ax.imshow(img, cmap=cmap, norm=norm)
        ax.axis("off")
        ax.set_title(
            f"Soilloss simulation – week {state['current_step']} / {state['n_steps']}"
        )

        # Clean right-side colorbar
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_ticks([-5, 0, 1, 3.5, 6.25, 8.75, 12.5, 17.5, 25])
        cbar.ax.set_yticklabels(labels)

        plt.tight_layout()
        plt.show()

# =========================
# 6. Step logic
# =========================
def do_step():
    if state["current_step"] >= state["n_steps"]:
        print("Конец симуляции за год.")
        return

    rain_mm = rain_slider.value
    state["cumulative_rain_mm"] += rain_mm

    # Add erosion proportional to rainfall fraction
    rain_fraction = rain_mm / ANNUAL_RAIN_MM
    state["current_map"] += rain_fraction * state["target"]

    # Advance model time
    state["current_step"] += 1

    # Optional: reset slider after an event
    rain_slider.value = 0.0

    info_label.value = (
        f"<b>Week:</b> {state['current_step']} / {state['n_steps']} | "
        f"<b>Total rain:</b> {state['cumulative_rain_mm']:.1f} mm"
    )

    update_plot()


def do_step10():
    for _ in range(10):
        do_step()
        if state["current_step"] >= state["n_steps"]:
            break


def do_reset(_=None):
    state["current_map"][:] = 0.0
    state["current_step"] = 0
    state["cumulative_rain_mm"] = 0.0
    rain_slider.value = 0.0
    info_label.value = "<b>Неделя:</b> 0 | <b>Осадков всего:</b> 0 mm"
    update_plot()

# =========================
# 7. Wire controls
# =========================
step_button.on_click(lambda b: do_step())
step10_button.on_click(lambda b: do_step10())
reset_button.on_click(do_reset)

# =========================
# 8. Display UI
# =========================
do_reset()

controls = widgets.VBox([
    widgets.HTML("<h3>🌧 LISEM bmi виджет</h3>"),
    rain_slider,
    widgets.HBox([step_button, step10_button, reset_button]),
    info_label
])

display(controls)
display(output_area)


In [ ]:
import os
from osgeo import gdal
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

import ipywidgets as widgets
from IPython.display import display, clear_output


папка_результатов = "/home/jovyan/work/LISEM/results/"
имя_файла = "soilloss.map"

def открыть_карту(path):
    ds = gdal.Open(path, gdal.GA_ReadOnly)
    arr = ds.GetRasterBand(1).ReadAsArray()
    ndv = ds.GetRasterBand(1).GetNoDataValue()
    if ndv is not None:
        arr = np.ma.masked_equal(arr, ndv)
    return arr

границы = [-9999, -10, 0, 2.5, 5, 7.5, 10, 15, 20, 9999]
ярлыки = [
    "денудация, >10", "0–10", "0–2.5", "2.5–5", "5–7.5",
    "7.5–10", "10–15", "15–20", ">20 аккумуляция"
]

цвета = [
    "#08306b", "#7fcdbb", "#f7fcb9", "#fee08b",
    "#fdae61", "#f46d43", "#d73027", "#a50026", "#67001f"
]

карта_цветов = ListedColormap(цвета)
норма = BoundaryNorm(границы, len(цвета))

путь = os.path.join(папка_результатов, имя_файла)
if not os.path.exists(путь):
    raise FileNotFoundError(f"Файл не найден: {путь}")

arr = открыть_карту(путь)
if isinstance(arr, np.ma.MaskedArray):
    маска = arr.mask
    конечная_карта = arr.filled(0.0).astype(float)
else:
    маска = None
    конечная_карта = arr.astype(float)

ny, nx = конечная_карта.shape


ГОДОВОЙ_ОСАДОК_ММ = 1000.0         # мм/год
ФОНОВАЯ_ЭРОЗИЯ_ММ_В_НЕДЕЛЮ = 0.5   # эквивалент очень слабой эрозии


состояние = {
    "цель": конечная_карта,
    "текущая_карта": np.zeros_like(конечная_карта),
    "неделя": 0,
    "всего_недель": 52,
    "накопленные_осадки_мм": 0.0
}


слайдер_осадки = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=100.0,
    step=1.0,
    description="Осадки (мм/нед):",
    continuous_update=False,
    layout=widgets.Layout(width="400px")
)

кнопка_шаг = widgets.Button(
    description="▶ Шаг",
    button_style="success",
    layout=widgets.Layout(width="120px")
)

кнопка_шаг10 = widgets.Button(
    description="▶▶ +10",
    button_style="info",
    layout=widgets.Layout(width="120px")
)

кнопка_сброс = widgets.Button(
    description="🔄 Сброс",
    button_style="warning",
    layout=widgets.Layout(width="120px")
)

метка_инфо = widgets.HTML(
    value="<b>Неделя:</b> 0 | <b>Осадков всего:</b> 0 мм"
)

область_вывода = widgets.Output()


def обновить_график():
    with область_вывода:
        clear_output(wait=True)

        img = состояние["текущая_карта"]
        if маска is not None:
            img = np.ma.array(img, mask=маска)

        fig, ax = plt.subplots(figsize=(8, 8))
        im = ax.imshow(img, cmap=карта_цветов, norm=норма)
        ax.axis("off")
        ax.set_title(
            f"Модель эрозии – неделя {состояние['неделя']} / {состояние['всего_недель']}"
        )

        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_ticks([-5, 0, 1, 3.5, 6.25, 8.75, 12.5, 17.5, 25])
        cbar.ax.set_yticklabels(ярлыки)

        plt.tight_layout()
        plt.show()


def сделать_шаг():
    if состояние["неделя"] >= состояние["всего_недель"]:
        print("✔ Симуляция завершена (52 недели).")
        return

    осадки = float(слайдер_осадки.value)

    состояние["накопленные_осадки_мм"] += осадки

    if осадки == 0:
        эффективные_осадки = ФОНОВАЯ_ЭРОЗИЯ_ММ_В_НЕДЕЛЮ
    else:
        эффективные_осадки = осадки

    доля_осадков = эффективные_осадки / ГОДОВОЙ_ОСАДОК_ММ
    состояние["текущая_карта"] += доля_осадков * состояние["цель"]

    состояние["неделя"] += 1

    слайдер_осадки.value = 0.0

    метка_инфо.value = (
        f"<b>Неделя:</b> {состояние['неделя']} / {состояние['всего_недель']} | "
        f"<b>Осадков всего:</b> {состояние['накопленные_осадки_мм']:.1f} мм "
        f"(фоновая эрозия работает при 0 мм)"
    )

    обновить_график()


def сделать_шаг10():
    for _ in range(10):
        сделать_шаг()
        if состояние["неделя"] >= состояние["всего_недель"]:
            break


def сброс(_=None):
    состояние["текущая_карта"][:] = 0.0
    состояние["неделя"] = 0
    состояние["накопленные_осадки_мм"] = 0.0
    слайдер_осадки.value = 0.0
    метка_инфо.value = "<b>Неделя:</b> 0 | <b>Осадков всего:</b> 0 мм"
    обновить_график()

# =========================
# 7. Привязка событий
# =========================
кнопка_шаг.on_click(lambda b: сделать_шаг())
кнопка_шаг10.on_click(lambda b: сделать_шаг10())
кнопка_сброс.on_click(сброс)

# =========================
# 8. Отображение интерфейса
# =========================
сброс()

интерфейс = widgets.VBox([
    widgets.HTML("<h3>Модель эрозии LISEM</h3>"),
    слайдер_осадки,
    widgets.HBox([кнопка_шаг, кнопка_шаг10, кнопка_сброс]),
    метка_инфо
])

display(интерфейс)
display(область_вывода)


In [ ]:


if общая_маска is not None:
    transport_masked = np.ma.array(transport_plus, mask=общая_маска)
    комбинированная_masked = np.ma.array(комбинированная, mask=общая_маска)
else:
    transport_masked = transport_plus
    комбинированная_masked = комбинированная

fig = plt.figure(figsize=(18, 8))

gs = gridspec.GridSpec(
    1, 3,
    width_ratios=[1, 1, 0.06],  
    wspace=0.05
)

ax_left = fig.add_subplot(gs[0, 0])
ax_right = fig.add_subplot(gs[0, 1])
ax_cbar = fig.add_subplot(gs[0, 2])

im1 = ax_left.imshow(transport_masked, cmap=карта_цветов, norm=норма)
ax_left.set_title("WaTEM-SEDEM")
ax_left.axis("off")

im2 = ax_right.imshow(комбинированная_masked, cmap=карта_цветов, norm=норма)
ax_right.set_title("WaTEM-SEDEM, ktc взят по расчетам LISEM")
ax_right.axis("off")

cbar = fig.colorbar(im2, cax=ax_cbar)
cbar.set_ticks([-5, 0, 1, 3.5, 6.25, 8.75, 12.5, 17.5, 25])
cbar.ax.set_yticklabels(ярлыки)

plt.tight_layout()
plt.show()


In [ ]:
import os
import subprocess

# Установить путь к LISEM
os.environ['OPENLISEM_EXECUTABLE'] = '/home/jovyan/work/LISEM/lisem-bin/Lisem'

# Запустить LISEM
lisem_path = os.environ['OPENLISEM_EXECUTABLE']
subprocess.run([lisem_path, 'config.yaml'])

In [ ]:
from bmi_openlisem import BmiOpenLisem

model = BmiOpenLisem()
model.initialize("/home/jovyan/work/LISEM/inputfiles/VNIIMZ_20m/maps/probnik_20m.run")



In [ ]:
while model.get_current_time() < model.get_end_time():
    model.update()
    print(model.get_current_time())

model.finalize()


In [ ]:
print("start:", model.get_start_time())
print("end:", model.get_end_time())
print("dt:", model.get_time_step())
print("n_steps ~", (model.get_end_time() - model.get_start_time()) / model.get_time_step())


In [ ]:
import time

t0 = time.time()
model.update()
t1 = time.time()
print("One update took", t1 - t0, "seconds")
print("Current time:", model.get_current_time())


# WATEM-SEDEM

In [ ]:
import os
os.getcwd()

In [ ]:
!git clone https://github.com/pihchikk/WaTEM-SEDEM-python

In [ ]:
!sudo apt-get update && sudo apt-get install -y libgdal34 gdal-bin


In [ ]:
os.chdir("WaTEM-SEDEM-python")

In [ ]:
!pip install -r requirements.txt


In [ ]:
!python src/run_watem.py -c config.yaml --mode hybrid
